In [11]:
import joblib
import pandas as pd
from sklearn import set_config
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import roc_auc_score
from pathlib import Path
import datetime as dt
import warnings
import time
import pytz
import json
import os

In [12]:
pst = pytz.timezone('America/Los_Angeles')
dt_str = dt.datetime.now(pst).strftime("%Y-%m-%d-%I_%M_%S_%p")
dt_str

'2026-08-08-05_06_01_PM'

In [13]:
experiment_config = {
    "experiment": {
        "id": f"{dt_str}_xgb",
        "model": "xgb",
        "type": "baseline",
        "description": "basic xgboost baseline",
    },

    "cv": {
        "strategy": "StratifiedKFold",
        "n_splits": 5,
        "shuffle": True,
        "random_state": 0
    },

    "params": {
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "n_estimators": 1500,
        "learning_rate": 0.1,
        "max_depth": 5,
        "tree_method": "hist",
        "enable_categorical": True,
        "early_stopping_rounds": 10
    }
}

In [14]:
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    data_path = "/kaggle/input/datasets/abhinavneelam/smartphone-addiction/data"
    output_path = "/kaggle/working/"
else:
    data_path = "../../data"
    output_path = "../../"

experiment_path = Path(output_path) / "experiments" / f"{dt_str}_{experiment_config["experiment"]["model"]}"
experiment_path.mkdir(parents=True, exist_ok=True)
experiment_path

WindowsPath('../../experiments/2026-08-08-05_06_01_PM_xgb')

In [15]:
with open(experiment_path / "config.json", "w") as f:
    json.dump(experiment_config, f, indent=4)

In [16]:
ss = pd.read_csv(f"{data_path}/raw/sample_submission.csv")
target_column = ss.columns[-1]
target_column

'addicted_label'

In [17]:
X = pd.read_csv(f"{data_path}/processed/train_features.csv")
X_test = pd.read_csv(f"{data_path}/processed/test_features.csv")
y = pd.read_csv(f"{data_path}/processed/train_labels.csv")

X.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 23 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   age                               662440 non-null  float64
 1   daily_screen_time_hours           595515 non-null  float64
 2   social_media_hours                557374 non-null  float64
 3   gaming_hours                      564548 non-null  float64
 4   work_study_hours                  639851 non-null  float64
 5   sleep_hours                       646889 non-null  float64
 6   notifications_per_day             623785 non-null  float64
 7   app_opens_per_day                 610659 non-null  float64
 8   weekend_screen_time               579306 non-null  float64
 9   gender                            662335 non-null  str    
 10  stress_level                      636221 non-null  float64
 11  academic_work_impact              647145 non-null  float64
 12 

In [ ]:
cat_cols = X.select_dtypes(include=["object", "string"]).columns

for frame in [X, X_test]:
    for col in cat_cols:
        frame[col] = frame[col].astype('category')

In [19]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 23 columns):
 #   Column                            Non-Null Count   Dtype   
---  ------                            --------------   -----   
 0   age                               662440 non-null  float64 
 1   daily_screen_time_hours           595515 non-null  float64 
 2   social_media_hours                557374 non-null  float64 
 3   gaming_hours                      564548 non-null  float64 
 4   work_study_hours                  639851 non-null  float64 
 5   sleep_hours                       646889 non-null  float64 
 6   notifications_per_day             623785 non-null  float64 
 7   app_opens_per_day                 610659 non-null  float64 
 8   weekend_screen_time               579306 non-null  float64 
 9   gender                            662335 non-null  category
 10  stress_level                      636221 non-null  float64 
 11  academic_work_impact              647145 non-null 

In [ ]:
kf = StratifiedKFold(n_splits=5, random_state=0, shuffle=True)

y_cv = pd.Series(index=y.index, dtype=float, name='predicted_proba')

fold_scores = []

start_time = time.time()

for train_index, valid_index in kf.split(X, y):
    X_train, X_valid = X.iloc[train_index], X.iloc[valid_index]
    y_train, y_valid = y.iloc[train_index], y.iloc[valid_index]

    model = XGBClassifier(**experiment_config["params"])
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)])

    y_pred = model.predict_proba(X_valid)[:, 1]

    y_cv.iloc[valid_index] = y_pred

    fold_auc_score = roc_auc_score(y_valid, y_pred)
    fold_scores.append(round(fold_auc_score, 5))

elapsed = time.time() - start_time

y_pred_df = y_cv.to_frame()
y_pred_df.to_csv(experiment_path / "oof.csv", index=False)

[0]	validation_0-auc:0.92204
[1]	validation_0-auc:0.92420
[2]	validation_0-auc:0.92727
[3]	validation_0-auc:0.92934
[4]	validation_0-auc:0.93012
[5]	validation_0-auc:0.93026
[6]	validation_0-auc:0.93020
[7]	validation_0-auc:0.93030
[8]	validation_0-auc:0.93045
[9]	validation_0-auc:0.93046
[10]	validation_0-auc:0.93108
[11]	validation_0-auc:0.93106
[12]	validation_0-auc:0.93106
[13]	validation_0-auc:0.93104
[14]	validation_0-auc:0.93106
[15]	validation_0-auc:0.93149
[16]	validation_0-auc:0.93204
[17]	validation_0-auc:0.93214
[18]	validation_0-auc:0.93230
[19]	validation_0-auc:0.93282
[20]	validation_0-auc:0.93314
[21]	validation_0-auc:0.93338
[22]	validation_0-auc:0.93363
[23]	validation_0-auc:0.93388
[24]	validation_0-auc:0.93398
[25]	validation_0-auc:0.93424
[26]	validation_0-auc:0.93453
[27]	validation_0-auc:0.93478
[28]	validation_0-auc:0.93502
[29]	validation_0-auc:0.93514
[30]	validation_0-auc:0.93534
[31]	validation_0-auc:0.93545
[32]	validation_0-auc:0.93573
[33]	validation_0-au

In [21]:
valid_auc_score = roc_auc_score(y, y_cv)

print("Validation AUC:", valid_auc_score)

Validation AUC: 0.9640059267219242


In [ ]:
metrics = {
    "experiment": dt_str + f"_{experiment_config["experiment"]["model"]}",
    "model": f"{experiment_config["experiment"]["model"]}",
    "cv": {
        "strategy": "StratifiedKFold",
        "n_splits": 5,
        "random_state": 0,
        "fold_scores": fold_scores,
        "mean": round(sum(fold_scores) / len(fold_scores), 5),
        "std": round(float(pd.Series(fold_scores).std(ddof=1)), 5)
    },
    "primary_metric": {
        "name": "auc",
        "value": round(valid_auc_score, 5)
    },
    "training": {
        "duration_seconds": round(elapsed, 2)
    }
}

with open(experiment_path / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

In [ ]:
best_iteration = model.best_iteration if hasattr(model, "best_iteration") else experiment_config["params"]["n_estimators"]

experiment_config["params"]["early_stopping_rounds"] = None
experiment_config["params"]["n_estimators"] = best_iteration

model = XGBClassifier(**experiment_config["params"])
model.fit(X, y)

joblib.dump(model, experiment_path / f"{experiment_config["experiment"]["model"]}.pkl")

['..\\..\\experiments\\2026-08-08-05_06_01_PM_xgb\\xgb.pkl']

In [ ]:
y_pred = model.predict_proba(X_test)[:, 1]
ss[target_column] = y_pred

ss.to_csv(experiment_path / f"{experiment_config["experiment"]["model"]}_submission.csv", index=False)
ss

,id,addicted_label
0,691369,0.000345
1,691370,0.092749
2,691371,0.033189
3,691372,0.011532
4,691373,0.000794
...,...,...
296297,987666,0.000000
296298,987667,0.095923
296299,987668,0.806904
296300,987669,0.253333
